In [2]:
import os
import uuid
import logging

from dotenv import load_dotenv
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings

print("GOOGLE_API_KEY set:", bool(os.getenv("GOOGLE_API_KEY")))
print("EMBEDDING_MODEL:", os.getenv("EMBEDDING_MODEL"))

GOOGLE_API_KEY set: True
EMBEDDING_MODEL: models/gemini-embedding-001


In [3]:
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)
logger = logging.getLogger(__name__)

In [4]:
def chunk_transcript(text: str, metadata: dict) -> list[dict]:
    """Split a transcript into overlapping chunks with metadata."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
    )
    texts = splitter.split_text(text)
    total = len(texts)
    return [
        {
            "id": str(uuid.uuid4()),
            "text": chunk,
            "metadata": {
                "video_id": metadata["video_id"],
                "title": metadata["title"],
                "creator": metadata["creator"],
                "engagement_rate": metadata["engagement_rate"],
                "chunk_index": i,
                "total_chunks": total,
            },
        }
        for i, chunk in enumerate(texts)
    ]

In [5]:
def get_embeddings_model() -> GoogleGenerativeAIEmbeddings:
    """Return a configured Gemini embeddings model."""
    model = os.getenv("EMBEDDING_MODEL")
    if not model:
        raise RuntimeError("EMBEDDING_MODEL is not set in the environment.")
    return GoogleGenerativeAIEmbeddings(model=model)

In [6]:
def get_collection() -> chromadb.Collection:
    """Return the persistent ChromaDB collection for video chunks."""
    chroma_path = os.getenv("CHROMA_PATH", "./backend/chroma_db")
    client = chromadb.PersistentClient(path=chroma_path)
    return client.get_or_create_collection("video_chunks")

In [7]:
def store_chunks(chunks: list[dict]) -> int:
    """Embed and store chunks in ChromaDB, replacing any existing chunks for the same video."""
    if not chunks:
        return 0

    collection = get_collection()
    embeddings_model = get_embeddings_model()

    video_id = chunks[0]["metadata"]["video_id"]

    existing = collection.get(where={"video_id": video_id}, include=[])
    if existing["ids"]:
        collection.delete(ids=existing["ids"])
        logger.info("Deleted %d existing chunks for video_id=%s", len(existing["ids"]), video_id)

    batch_size = 100
    stored = 0
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i : i + batch_size]
        texts = [c["text"] for c in batch]
        embeddings = embeddings_model.embed_documents(texts)
        collection.add(
            ids=[c["id"] for c in batch],
            documents=texts,
            embeddings=embeddings,
            metadatas=[c["metadata"] for c in batch],
        )
        stored += len(batch)
        logger.info("Stored batch %d-%d", i, i + len(batch) - 1)

    return stored

In [8]:
def search_chunks(query: str, video_id: str | None = None, k: int = 4) -> list[dict]:
    """Embed a query and return the top-k matching chunks from ChromaDB."""
    embeddings_model = get_embeddings_model()
    query_embedding = embeddings_model.embed_query(query)

    collection = get_collection()
    where = {"video_id": video_id} if video_id else None

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )

    hits = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        hits.append({"text": doc, "metadata": meta, "distance": dist})
    return hits

In [9]:
SAMPLE_TRANSCRIPT = """
Large language models have transformed natural language processing by enabling zero-shot and few-shot learning
across a wide variety of tasks. Rather than fine-tuning a separate model for every task, a single large model
can be prompted with instructions and examples to solve reading comprehension, translation, summarization,
and more. This flexibility comes from scaling: as model size, data, and compute grow, emergent capabilities
appear that were absent in smaller models. Researchers have observed that capabilities such as chain-of-thought
reasoning, arithmetic, and multi-step problem solving emerge at scale without any explicit training signal
for those behaviors. The transformer architecture, with its self-attention mechanism, is the backbone of
these models. Attention allows every token to directly attend to every other token in the sequence, capturing
long-range dependencies more efficiently than recurrent networks. Positional encodings inject sequence order
into the attention computation since the architecture itself is permutation-invariant. Training involves
next-token prediction on enormous corpora from the internet, books, and code repositories. The resulting
models encode broad factual and procedural knowledge in their weights, enabling them to assist with coding,
writing, analysis, and question answering without task-specific supervision.
"""

SAMPLE_METADATA = {
    "video_id": "test_video_001",
    "title": "Understanding Large Language Models",
    "creator": "AI Research Channel",
    "engagement_rate": 0.087,
}

chunks = chunk_transcript(SAMPLE_TRANSCRIPT, SAMPLE_METADATA)
print(f"Created {len(chunks)} chunks")
for c in chunks:
    print(f"  chunk {c['metadata']['chunk_index']}: {len(c['text'])} chars")

Created 2 chunks
  chunk 0: 978 chars
  chunk 1: 503 chars


In [10]:
stored_count = store_chunks(chunks)
print(f"Stored {stored_count} chunks")

2026-05-19 02:24:13,600 INFO Deleted 2 existing chunks for video_id=test_video_001
2026-05-19 02:24:14,399 INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2026-05-19 02:24:14,489 INFO Stored batch 0-1


Stored 2 chunks


In [11]:
results = search_chunks("how does attention mechanism work", video_id="test_video_001", k=3)

print(f"Retrieved {len(results)} results\n")
for i, hit in enumerate(results):
    print(f"--- Result {i + 1} (distance={hit['distance']:.4f}) ---")
    print(f"Chunk {hit['metadata']['chunk_index']} of {hit['metadata']['total_chunks']}")
    print(f"Video: {hit['metadata']['title']} by {hit['metadata']['creator']}")
    print(hit["text"][:300])
    print()

2026-05-19 02:24:15,619 INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


Retrieved 2 results

--- Result 1 (distance=0.5704) ---
Chunk 1 of 2
Video: Understanding Large Language Models by AI Research Channel
long-range dependencies more efficiently than recurrent networks. Positional encodings inject sequence order
into the attention computation since the architecture itself is permutation-invariant. Training involves
next-token prediction on enormous corpora from the internet, books, and code repositor

--- Result 2 (distance=0.6059) ---
Chunk 0 of 2
Video: Understanding Large Language Models by AI Research Channel
Large language models have transformed natural language processing by enabling zero-shot and few-shot learning
across a wide variety of tasks. Rather than fine-tuning a separate model for every task, a single large model
can be prompted with instructions and examples to solve reading comprehension, 



In [ ]:
REAL_TRANSCRIPT = """
- Could the CIA really
track your heartbeat from kilometers away? On April 3rd, 2026, Iranian forces shot down
an American fighter plane just over Isfahan. Inside were a pilot and
a weapon system officer, and both ejected successfully. The US forces located the pilot quickly and rescued him only seven
hours after the crash, but they couldn't rescue
the weapon system officer. He landed elsewhere deep
within hostile territory. And, worst of all, he was injured. With the Iranian forces on his tail, the officer needed to hide quickly, so he disappeared into the mountains. to rescue an aviator buried
deep behind enemy lines. - Fortunately, the officer
had a rescue beacon that could signal his location to the US. The problem was that he
not only had to step out of his hiding spot to transmit the signal, Iran could potentially intercept
it and get to him first. So he could only use the beacon sparingly. With enemy forces getting
closer every hour, how is the US going to
pinpoint his location in the middle of a desert. - [Gregor] A blind sweep of
the entire area could take days or even weeks, but, surprisingly, just
40 hours after the crash, the US announced the officer was rescued. - Still invisible to the
enemy, but not to the CIA. - So how did they do it? Well, according to a
New York Post article, the CIA deployed a futuristic
device to rescue him. Reportedly they were able
to detect the magnetic field produced by his heartbeat
from kilometers away. Such a device would have to
overcome the magnetic signatures from other soldiers, vehicles,
and animals in the region, let alone Earth's magnetic field. It's like listening for
a murmur in a crowd. So the technology was
appropriately called Ghost Murmur. Immediately this kicked
off a media frenzy. - Ghost Murmur. - This is science fiction.
- Ghost Murmur. - Did you hear about what
the CIA tech that they have called the Ghost Murmur. - Called the Ghost Murmur. - All of this sounds too good to be true, and there seemed to be no other sources beyond this New York Post article. So we dug deep to find out
whether this supposed technology really exists and what its limits are. - I very rarely believe things
I read in the New York Post. - Okay. - I find it extremely
difficult to believe. - Many of these researchers
in the NV diamond area, are having to sign NDAs. - The fact that the CIA is involved in leveraging technologies consistent with their
mission and their charter. - So is Ghost Murmur fact or fiction? There are two lines in
this New York Post article that hint at what this device can be. First, normally this signal is so weak that it can only be measured
in a hospital setting with sensors pressed
nearly against the chest, the source said. But advances in a field known
as quantum magnetometry, specifically sensors built
around microscopic defects in synthetic diamonds, have apparently made it possible to detect these signals at
dramatically greater distances. We're gonna break this down bit by bit. First, does the heart actually create detectable magnetic fields? Second, what are these synthetic diamonds that could potentially detect them? And third, is it all
possible at these distances? Let's start with a heart. Now, if you type heart magnetic
field into Google Images, you will get a bunch of
questionable-looking graphs. So is it a real thing? Well, whenever current
flows through a conductor, it generates a magnetic field around it. And since our bodies run
on electrical impulses traveling through neurons, our tissues and organs generate
faint magnetic signals. But because the heart muscles
fire in a coordinated way, the magnetic field they produce is the strongest in the body. It's around 50 to 100 pico Teslas, 10 to 100 times more than the next strongest
field produced by the brain. But even then, this is
still a million times weaker than Earth's magnetic field. So it's no surprise that we only detected the magnetic field of the heart in 1963. It had to be done in a remote field away from the magnetic noise produced by lab equipment,
elevators, and cars. And the setup had to be incredibly still. Even the slightest vibration of the detector would
corrupt the measurement, not something that could
work on a helicopter or a military drone. But, pretty soon,
magnetometers got better. By the 1970s, we got superconducting
quantum interference devices, or SQUIDs. These magnetometers were
incredibly sensitive, detecting fields as weak
as a few femto Tesla. To no surprise, the US military quickly
strapped these SQUIDs to planes and helicopters, and they tried to use them to detect large magnetic signatures like submarines in the ocean. But the project never really picked up. Nonetheless, SQUIDs also made it easier to detect the heart's magnetic
field, but with a key caveat. - They typically need to be operated under a very tightly
controlled conditions, often inside of shielded rooms. They can't handle large dynamic
range of background fields and electromagnetic interference. - Future magnetometers offered solutions, but they also had their own drawbacks. There were always issues
with either shielding or sensitivity or dynamic range
that made them impractical for detecting heartbeats out in the field. Until the 1990s when physicists started looking into diamonds that
might eventually be able to sense magnetic fields while potentially getting
around the drawbacks. - These new quantum magnetometers work at room temperature operation, and that's what's really
exciting about them. They're also a solid state sensor, which, you know, can be very
practical for some applications and can be made into like a
more robust kind of sensor. - These are the diamonds mentioned in the New York Post article. So how do they work? Now, the overall coverage of the story is actually very interesting because this tech is
supposedly classified. There's a lot of uncertainty about whether what's being
reported is actually real. And depending on which
outlet you see first, you might arrive at completely
different conclusions. For example, if you saw
this headline first, you might be really impressed
with the technology, but this one you'd be
a bit more skeptical, and the last one might
not even make you care about the tech. So three headlines, three
completely different conclusions, and that's where today's
sponsor Ground News comes in. They're a website and an app designed to make reading the news easier and more data-driven. Each day they round up stories
from thousands of outlets all around the world and give you a visual
breakdown of the story, including its political
leaning, factuality rating, and even ownership, all backed by ratings from three independent
media-monitoring organizations. In case of this Ghost Murmur story, you can see that it's been
reported on by 65 sources, but the coverage is pretty lopsided. So if you only get your news
from right-leaning outlets, there's a chance you've seen the story. But if you only get it from the left, there's a chance you haven't. And below that, you can also
see the factuality rating and the ownerships of the sources. They also have this blind spot feature, which highlights stories
like the Ghosts Murmur one that were under reported on by either side of the political spectrum. Staying up to date and well-informed with accurate information has
almost become a full-time job. And that's where Ground News's
approach is so valuable. They make reading the news
more digestible and accessible, and you can see all of the
information within moments, not hours. So if you, like us at Veritasium, care about getting to the truth, you can go to ground.news/ve
or scan this QR code and our link will get you
40% off their vantage plan. So I wanna thank Ground News for sponsoring this part of the video, and now let's go figure out how those diamonds actually
detect magnetic fields. Well, for something to
function as a magnetometer, it needs to respond to a magnetic field in a way that we can detect. Now, a pure diamond is
just an ordered lattice of carbon atoms, so it doesn't react to magnetic
fields in a meaningful way. But this changes when you start adding
defects to the lattice. You can replace one of the
carbon atoms in the lattice with, say, a nitrogen. And if you remove one of the
carbons next to it completely, well, that creates a vacancy. This defect is called a nitrogen
vacancy or an NV center. And these NV centers
become particularly useful when they trap two unpaired electrons. That's because electrons have this intrinsic property called spin. A simple and flawed analogy is that spin is kind of
like a tiny bar magnet that gives electrons their
own magnetic signature and it can point either up or down. So when an electron is exposed to an external magnetic field, this magnetic signature
will either align itself with the field or against it. Also note that particles
that have no net spin will not respond to an
external magnetic field. Now, the two trapped electrons
can arrange their spins in the following way. They could both point up, which would give you a
spin magnetic number of 1. They could both point down
for a quantum number of -1, or they could point in opposite directions for a quantum number of 0. We'll denote this spin magnetic
quantum number with ms, and it essentially acts as a bar magnet for the whole NV center. And just like for individual electrons, this bar magnet is also sensitive to external magnetic fields. So now that we've created a diamond that responds to magnetic
fields, the challenge is, how do we measure this
response to detect a heartbeat? Okay, so I reached out to experts to try and figure this out, but basically out of the 20
or so emails that I sent, I only got a few responses, but then I got an impromptu
call from one of these experts who said that many of these researchers in the NV diamond area
are having to sign NDAs. This is getting a lot more interesting. But with how secretive everyone was being, I had to use publicly available research. And I think the key idea is to figure out how diamonds respond to magnetic fields, you have to use light. When you shine light at an atom, the atom can either absorb
the light or ignore it. An absorbed photon of light
will excite an electron within the atom to a higher energy level. You can think of these energy
levels as discrete platforms that the electrons can jump between, just like in a video game. All atoms of the same
element, for example, carbon, have the same energy levels when the atoms are far enough apart. But when you bring them together, like inside a diamond lattice,
their energy levels shift. They come together to form a series of closely spaced energy
levels or an energy band. Now, an electron will
only ever absorb a photon if the photon has enough energy to move the electron across
the gap to a higher band. This gap between the last
electron occupied band and the first empty band above
it is called the band gap. In a pure diamond, it's big. It's around 5.5 electron volts, which means that only ultraviolet
photons have enough energy to excite the electrons. All lower energy light,
including visible light, will mostly be ignored by the
diamond and just pass through. This is why a perfect pure
diamond is transparent. It doesn't absorb any of the light, but if you start adding defects, they disrupt that organized lattice. The defects unlock
different energy levels. Secret platforms within this band gap for nearby electrons to jump to. A boron defect, for example, creates a low energy level
at only 0.37 electron volts. This is a platform that
electrons can jump to by absorbing infrared or red light. And with enough boron defects, a significant amount of red
light gets absorbed this way. The rest of the visible spectrum
mostly makes it through. So without this red, the
diamond appears blue. Similarly, a nitrogen
vacancy defect unlocks other secret platforms. And these can help us detect how the NV center responds
to a magnetic field. At first glance, it looks
like the nitrogen vacancy generates a few unique levels, and these are exclusive to the two unpaired electrons
trapped within the defect. But if you look closer at,
for example, the lowest level, you'll notice that it actually contains three closely spaced but
separate energy levels. The second and third levels are
actually at the same height, but will draw them separately. And the fact that there are
three isn't a coincidence. Remember, the NV center can
adopt one of three ms numbers, 0, -1, or 1, depending on how the spins of the two electrons inside are arranged. If you think of the two
spins as bar magnets, the most relaxed, lowest energy
way for them to sit is this, one pointing up and the other down. This is analogous to the ms = 0 state, which is why it has the lowest energy. Forcing both magnets to point
down together or up together, like the ms = 1 or -1
states requires more energy. They oppose you. So these two states are at an equal, slightly
higher energy sub level. These differences are tiny. Jumping from the 0 to
the +/-1 levels requires only a small amount of energy. A microwave photon of 10.4
centimeters will be enough. Now, these secret energy platforms of NV centers were mapped out by the '90s. But for a long time, no one thought to use
them as magnetometers. - There was a decade before
the light bulb went on for a bunch of us to think
about them as sensors. It takes a mindset switch
to think differently. And once you do, you realize, "Oh my goodness, this could be useful." - So let's apply a magnetic
field to this diamond and see what happens. If we slowly turn up the field strength, you'll see that the +1 and -1
levels are starting to shift. To understand why, we can use a compass needle as an analogy. Naturally, a compass
needle will align itself with the magnetic field of the Earth. And this is the lowest
energy relaxed state that the needle can be in. And it's analogous to the
behavior of the system in the ms = -1 state, which is why its energy
level drops slightly. But if ms = 1, the system flips. This would be like
taking an external magnet and applying it to this compass needle, flipping it 180 degrees. And then if I'm careful
in retracting this magnet, I can actually get the
needle to stay in this place. And this is the highest
energy this needle can have sitting in this unstable position. Now if I tap it, it will actually go back. It's possible, but it's
a higher energy state. So the ms = 1 level slightly rises. And if ms = 0, it doesn't
react to the field, which is why this level hasn't changed. Now, if you keep turning up
the magnetic field strength, you'll notice that the levels
get further and further apart. This phenomenon is
called Zeeman splitting. It's described by a simple formula that gives you a direct link
between the energy split and the magnetic field strength. So in the presence of a
periodic magnetic field like that generated by the heart, we would theoretically
see a rhythmic separation of these lines. And these two energy levels absorb light at two different microwave wavelengths, which change depending on
the strength of the field. So what we can actually measure is which microwave wavelengths
the diamond is absorbing. When there's no magnetic field, the levels are fused and
produce a single absorption line at the wavelength of 10.4 centimeters. But when there's an
external magnetic field, they produce two separate
absorption lines. Now, we've simplified it a bit, but by measuring how spaced
apart these lines are, you get the field strength. This is how an NV diamond
magnetometer works. Was the diamond magnetometer ever used to detect a heartbeat? - So what has been done
for sure is, you know, for certain is detection of magnetic fields generated by neurons. Yeah, I mean, which is
to some extent connected to the heartbeat question. Neuron activity, to my knowledge, has been first seen in 2015. - Okay, well, that's,
wow, that's impressive. So could it pick up a
heartbeat from kilometers away? Well, in 2022, researchers were able to pick up the magnetic
field of a rat's heart, but it was done using a thoracotomy, which means the rat's chest was open and the diamond was less
than two millimeters away from the heart. Okay, but the human heart produces a stronger magnetic field. And the tech, the CIA might have deployed, could be decades ahead of
what is publicly known. You're former CIA operations officer and you've worked in
the Middle East, right? So what was your first
reaction to this news? - Well, the fact that the CIA's involved in leveraging technology is consistent with their mission and their charter. But I defer to smarter
people in engineering and science like yourself to figure out the exact
technique that might be used. But the processes, of
course, are consistent for what we've done. - We can't say for sure
whether the CIA has this tech, but we can use physics to estimate how sensitive it would need to be. Well, the strength of a
magnetic field falls off with the cube of the
distance from the source. So if the magnetic field of the heart when measured at the
chest is 50 pico Tesla, or 5 times 10 to the -11 Tesla, well, then just 100 meters away, this falls off by a factor of a billion to 5 times 10 to the -20 Tesla. And at 50 to 100 kilometers, this could drop to as little
as 10 to the -30 Tesla. - The most sensitive measurement ever made at the frequencies that the
human heartbeat work at is at the 10 to the -15 Tesla level. And that's in a shielded room. So you'd need a system that is 15 orders of
magnitude more sensitive than the superconducting
quantum interference devices and 18 orders of magnitude more sensitive than diamond NV sensors. - 18 orders of magnitude is a
lot, sounds quite unfeasible. - It's not like the hills of
Iran are devoid of animal life. They also have heartbeats and possibly larger hearts than humans. - There's also the
magnetic field of the drone or the helicopter the
device might be mounted on. And, finally, there's the fact that a magnetic field of 10
to the -30 Tesla is weaker than a magnetic field an
electron will give you a meter away. - The New York Post is
notoriously a very good place for amusing fiction. - On the day before we saw this article, we also saw an article about how they had a technology
where it was a beeper. and that was one way that
they were able to detect him. And these are things that
we know about already. We also know that there may well have been other intelligence methods used, but this isn't really necessary. - And then that brings
you to the point of like, why would they make this up? - I think New York Post is
known to print a lot of stuff. - I'm not a real fan of the credibility of the press to report things, having seen reality and then
seeing what the press reports. - How about deception stories and like false narratives
published by not only the CIA, but other agencies? Is this often the case? - If you look historically, the idea of fooling your enemy, particularly when you have
some vulnerability to protect, goes back thousands of years. - During World War II, German bombers frequently
attacked Britain at night. And to retaliate, the British
would fly up to intercept and somehow kept finding
these bombers in the dark and would attack them. Now, UK officials told the press that the pilots were able to do this because they ate a lot of carrots, which was improving their vision at night. But some experts believe that this was actually a cover story meant to distract the Germans from the fact the British
installed radars on their planes. - Is that where the idea that carrots give you
good eyesight comes from? - I think so. I think this is the original,
yeah, the original myth. - That is crazy.
- Yeah. - But, okay, there's one
thing that still bothers me. If this is likely fake, then why is everyone saying no comment and declining to talk? There must be something here, you know? - Well, these NV centers
and diamonds are also used for quantum computing. But the more interesting, probably confidential
way they could be used is as navigation devices. The Earth's magnetic field
creates a unique pattern all across the globe. - Then to place an object into this map, and if you know all of these perturbations and in homogeneities, you can infer where you are without having any GPS reception anymore. - With the rise of GPS spoofing and jamming over the last couple of years, that could be incredibly powerful. So NV magnetometers do exist
with these synthetic diamonds, and they do have potential
military applications. It's just that detecting
heartbeats kilometers away probably isn't one of them. I wanna give a shout-out to a great Scientific American article on this story by Deni Béchard that initially expressed
skepticism in Ghost Murmur as a potential technology. It's a great read and you check it out.
"""
SAMPLE_METADATA = {
    "video_id": "test_video_001",
    "title": "Understanding Large Language Models",
    "creator": "AI Research Channel",
    "engagement_rate": 0.087,
}

REAL_METADATA = {'video_id': 'SVTPv4sI_Jc', 'title': "The CIA's Worst New Tech Idea?", 'creator': 'Veritasium', 'engagement_rate': 3.3747}

chunks = chunk_transcript(REAL_TRANSCRIPT, REAL_METADATA)
print(f"Created {len(chunks)} chunks")
for c in chunks:
    print(f"  chunk {c['metadata']['chunk_index']}: {len(c['text'])} chars")

Created 26 chunks
  chunk 0: 977 chars
  chunk 1: 969 chars
  chunk 2: 997 chars
  chunk 3: 952 chars
  chunk 4: 981 chars
  chunk 5: 996 chars
  chunk 6: 960 chars
  chunk 7: 963 chars
  chunk 8: 997 chars
  chunk 9: 911 chars
  chunk 10: 979 chars
  chunk 11: 784 chars
  chunk 12: 914 chars
  chunk 13: 947 chars
  chunk 14: 942 chars
  chunk 15: 967 chars
  chunk 16: 948 chars
  chunk 17: 996 chars
  chunk 18: 991 chars
  chunk 19: 995 chars
  chunk 20: 913 chars
  chunk 21: 982 chars
  chunk 22: 985 chars
  chunk 23: 993 chars
  chunk 24: 999 chars
  chunk 25: 373 chars


In [18]:
stored_count = store_chunks(chunks)
print(f"Stored {stored_count} chunks for video_id={REAL_METADATA['video_id']}")

2026-05-19 02:56:35,970 INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2026-05-19 02:56:36,337 INFO Stored batch 0-25


Stored 26 chunks for video_id=SVTPv4sI_Jc


In [19]:
results = search_chunks("how much magnetic field is consider weaker", video_id="SVTPv4sI_Jc", k=30)

print(f"Retrieved {len(results)} results\n")
for i, hit in enumerate(results):
    print(f"--- Result {i + 1} (distance={hit['distance']:.4f}) ---")
    print(f"Chunk {hit['metadata']['chunk_index']} of {hit['metadata']['total_chunks']}")
    print(f"Video: {hit['metadata']['title']} by {hit['metadata']['creator']}")
    print(hit["text"][:300])
    print()

2026-05-19 02:56:40,929 INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


Retrieved 26 results

--- Result 1 (distance=0.6291) ---
Chunk 4 of 26
Video: The CIA's Worst New Tech Idea? by Veritasium
questionable-looking graphs. So is it a real thing? Well, whenever current
flows through a conductor, it generates a magnetic field around it. And since our bodies run
on electrical impulses traveling through neurons, our tissues and organs generate
faint magnetic signals. But because the heart musc

--- Result 2 (distance=0.6383) ---
Chunk 17 of 26
Video: The CIA's Worst New Tech Idea? by Veritasium
levels are starting to shift. To understand why, we can use a compass needle as an analogy. Naturally, a compass
needle will align itself with the magnetic field of the Earth. And this is the lowest
energy relaxed state that the needle can be in. And it's analogous to the
behavior of the system in t

--- Result 3 (distance=0.6404) ---
Chunk 18 of 26
Video: The CIA's Worst New Tech Idea? by Veritasium
react to the field, which is why this level hasn't changed. Now, if y